In [ ]:
import math

class FoodTruckAgent:
    def __init__(self):
        # Uncertain world state:
        # There is a festival today or there is no festival.
        self.states = ["festival", "no_festival"]

        # Prior belief about the world
        self.prior = {
            "festival": 0.35,
            "no_festival": 0.65
        }

        # Available direct actions
        self.actions = ["go_downtown", "go_park"]

        # Utility table: profit/loss for each action under each state
        # Utilities are in arbitrary utility points (can be thought of as dollars)
        self.utilities = {
            "go_downtown": {
                "festival": 140,
                "no_festival": 20
            },
            "go_park": {
                "festival": 70,
                "no_festival": 90
            }
        }

        # Information-gathering action:
        # Buy a market report before choosing where to go
        self.info_action_name = "buy_market_report"
        self.info_cost = 8

        # Report accuracy model
        # P(report = signal | true state)
        self.report_model = {
            "festival": {
                "festival_signal": 0.85,
                "no_festival_signal": 0.15
            },
            "no_festival": {
                "festival_signal": 0.20,
                "no_festival_signal": 0.80
            }
        }

    def expected_utility(self, action, beliefs=None):
        if beliefs is None:
            beliefs = self.prior
        return sum(beliefs[state] * self.utilities[action][state] for state in self.states)

    def best_action(self, beliefs=None):
        if beliefs is None:
            beliefs = self.prior
        scores = {action: self.expected_utility(action, beliefs) for action in self.actions}
        best = max(scores, key=scores.get)
        return best, scores

    def signal_probability(self, signal):
        total = 0.0
        for state in self.states:
            total += self.prior[state] * self.report_model[state][signal]
        return total

    def posterior(self, signal):
        denom = self.signal_probability(signal)
        post = {}
        for state in self.states:
            numer = self.prior[state] * self.report_model[state][signal]
            post[state] = numer / denom
        return post

    def expected_utility_with_information(self):
        signals = ["festival_signal", "no_festival_signal"]
        total_eu = 0.0
        details = {}

        for signal in signals:
            p_signal = self.signal_probability(signal)
            post = self.posterior(signal)
            best_after_signal, action_scores = self.best_action(post)
            best_signal_eu = action_scores[best_after_signal]

            details[signal] = {
                "probability_of_signal": p_signal,
                "posterior_beliefs": post,
                "action_scores_after_signal": action_scores,
                "best_action_after_signal": best_after_signal,
                "expected_utility_after_signal": best_signal_eu
            }

            total_eu += p_signal * best_signal_eu

        total_eu_minus_cost = total_eu - self.info_cost
        return total_eu_minus_cost, details

    def value_of_information(self):
        best_now, scores_now = self.best_action(self.prior)
        eu_without_info = scores_now[best_now]

        eu_with_info, info_details = self.expected_utility_with_information()

        voi = eu_with_info - eu_without_info

        return {
            "best_action_without_information": best_now,
            "expected_utilities_without_information": scores_now,
            "eu_without_information": eu_without_info,
            "eu_with_information_minus_cost": eu_with_info,
            "value_of_information": voi,
            "information_worth_buying": voi > 0,
            "information_details": info_details
        }

    def choose(self):
        voi_result = self.value_of_information()

        if voi_result["information_worth_buying"]:
            final_choice = self.info_action_name
            rationale = (
                f"The agent should first take the information-gathering action "
                f"because its net expected utility ({voi_result['eu_with_information_minus_cost']:.2f}) "
                f"is higher than acting immediately ({voi_result['eu_without_information']:.2f})."
            )
        else:
            final_choice = voi_result["best_action_without_information"]
            rationale = (
                f"The agent should act immediately and choose '{final_choice}' "
                f"because paying for information does not improve expected utility enough."
            )

        return final_choice, rationale, voi_result


def print_report(agent):
    print("=" * 70)
    print("SIMPLE RATIONAL AGENT UNDER UNCERTAINTY")
    print("=" * 70)

    print("\n1) Prior beliefs about the world")
    for state, prob in agent.prior.items():
        print(f"   P({state}) = {prob:.2f}")

    print("\n2) Utility table")
    for action in agent.actions:
        print(f"   Action: {action}")
        for state in agent.states:
            print(f"      U({action}, {state}) = {agent.utilities[action][state]}")

    print("\n3) Expected utility of acting immediately")
    best_now, scores_now = agent.best_action()
    for action, eu in scores_now.items():
        print(f"   EU({action}) = {eu:.2f}")
    print(f"   Best immediate action = {best_now}")

    print("\n4) Information-gathering action")
    print(f"   Action name: {agent.info_action_name}")
    print(f"   Cost of information: {agent.info_cost}")
    print("   Report accuracy model:")
    for true_state in agent.states:
        for signal, prob in agent.report_model[true_state].items():
            print(f"      P({signal} | {true_state}) = {prob:.2f}")

    eu_with_info, info_details = agent.expected_utility_with_information()

    print("\n5) What happens after observing information")
    for signal, data in info_details.items():
        print(f"\n   Signal observed: {signal}")
        print(f"   P(signal) = {data['probability_of_signal']:.4f}")
        print("   Posterior beliefs:")
        for state, prob in data["posterior_beliefs"].items():
            print(f"      P({state} | {signal}) = {prob:.4f}")
        print("   Expected utilities after this signal:")
        for action, eu in data["action_scores_after_signal"].items():
            print(f"      EU({action} | {signal}) = {eu:.2f}")
        print(f"   Best action after signal: {data['best_action_after_signal']}")
        print(f"   Best conditional EU: {data['expected_utility_after_signal']:.2f}")

    voi_result = agent.value_of_information()

    print("\n6) Value of information")
    print(f"   Best EU without information = {voi_result['eu_without_information']:.2f}")
    print(f"   Best EU with information (minus cost) = {voi_result['eu_with_information_minus_cost']:.2f}")
    print(f"   VOI = {voi_result['value_of_information']:.2f}")
    print(f"   Information worth buying? {voi_result['information_worth_buying']}")

    final_choice, rationale, _ = agent.choose()

    print("\n7) Final rational choice")
    print(f"   Agent chooses: {final_choice}")
    print(f"   Rationale: {rationale}")
    print("=" * 70)


if __name__ == "__main__":
    agent = FoodTruckAgent()
    print_report(agent)

## Answers to the Required Questions

### How did probabilities affect decisions?

Probabilities affected decisions by determining the agent’s belief about how crowded each location was likely to be at the decision time. Since waiting cost depends on whether a location is busy, the probability of busyness directly influenced expected utility.

A location with a high probability of being busy had a larger expected waiting cost, which reduced its utility. A location with a lower probability of being busy became more attractive, even if it was not the closest option. In this way, probabilities changed the ranking of actions by changing the expected cost of each one.

More broadly, probabilities allowed the agent to reason under uncertainty instead of assuming that the world was fully known. Rather than asking “Is this location busy?”, the model asked “How likely is it to be busy?” and made the decision accordingly.

### How sensitive were decisions to utilities?

Decisions were highly sensitive to utilities because the final choice was based on whichever action had the highest expected utility. Even modest changes in the utility components—walking time, waiting time, mismatch penalty, or information cost—could change which action was optimal.

For example:
- if the waiting penalty for a crowded location increased, that location would become less attractive;
- if walking time to a preferred location decreased, that location could become the best option;
- if mismatch penalties changed, the ranking of locations could shift even when busyness probabilities stayed the same.

This shows that the rational choice depends not only on probabilities, but also on how outcomes are valued. Utility functions are therefore very important, because they determine what counts as “better” for the agent.

### When was information worth paying for?

Information was worth paying for when the expected benefit of sensing exceeded the information cost. In this model, that happened when observing whether StudentCenter was busy could meaningfully improve the agent’s later action choice.

In particular, information became valuable when:
- the prior uncertainty about the sensed location was substantial,
- the sensed location strongly affected the utility of candidate actions,
- and the sensor was accurate enough that the updated belief would change the decision in a useful way.

If the prior belief already strongly favored one action, then sensing was usually not worth it because the observation would not change much. Likewise, if the information cost was too high, then even useful information was not worth purchasing.

So, information was worth paying for only when it had enough expected decision value to overcome its cost.